# Closed-form RNNs for discrete $SE(3)$

This notebook constructs a QuadraticRNN for

$$G=\mathbb{Z}_n^3\rtimes O,$$

where $O$ is the 24-element group of orientation-preserving cubic rotations. A vector $x\in\mathbb{R}^{|G|}$ is equivalently a scalar signal $x:G\to\mathbb{R}$, and the network realizes the regular left action

$$(g\cdot x)[h]=x[g^{-1}h].$$

We make two distinct claims:

1. **Exact small case:** all irreps at $n=2$ reproduce translation, rotation, and mixed composition to floating-point precision.
2. **Truncated spatial case:** selected Fourier blocks at $n=3$ trade exact reconstruction for a much smaller hidden state.

No weights are learned.

In [ ]:
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.discrete_se3_geometry import (
    decode_pose,
    gaussian_landmark,
    orientation_marginal,
    peaked_orientation_weights,
    periodic_distance_squared,
    plot_orientation_marginal,
    plot_orthogonal_slices,
    plot_trajectory,
    rotation_error,
    spatial_marginal,
    transformed_pose,
)
from src.finite_group_rnn import (
    build_finite_group_rnn,
    hidden_width,
    random_invertible_encoding,
    rollout,
    select_irreps_by_power,
)
from src.groups import DiscreteSE3Group

np.set_printoptions(precision=3, suppress=True)

## 1. Group and representation structure

Write an element as $g=(t,r)$, with $t\in\mathbb{Z}_n^3$ and $r\in O$. If $R_r$ is the signed permutation matrix for rotation $r$, then

$$(t_1,r_1)(t_2,r_2)=(t_1+R_{r_1}t_2,\ r_1r_2).$$

Elements are flattened with rotation as the most significant coordinate:

$$\operatorname{idx}(x,y,z,r)=rn^3+xn^2+yn+z.$$

Irreps are constructed by the little-group method: cubic rotations act on translation frequencies $k\in\mathbb{Z}_n^3$, and stabilizer irreps are induced to the full semidirect product. The implementation checks $\sum_\rho d_\rho^2=|G|$.

In [ ]:
G_exact = DiscreteSE3Group(n=2)
irreps_exact = G_exact.irreps()
dimension_counts = Counter(irrep.dim for irrep in irreps_exact)
exact_width = sum(hidden_width(irrep) for irrep in irreps_exact)

print(f"|G| = {G_exact.order}")
print(f"number of irreps = {len(irreps_exact)}")
print("irrep dimension counts =", dict(sorted(dimension_counts.items())))
print("Peter–Weyl sum =", sum(irrep.dim**2 for irrep in irreps_exact))
print("all-irrep hidden width =", exact_width)
assert sum(irrep.dim**2 for irrep in irreps_exact) == G_exact.order

## 2. Closed-form QuadraticRNN

For each irrep,

$$\widehat{x}(\rho)=\sum_{g\in G}x(g)\rho(g)^\dagger.$$

Let $\phi(z)=\operatorname{ReLU}(z)^2$. For relative drives $g_1,\ldots,g_T$,

$$h_1=\phi(W_{\rm in}x_{\rm allo}+W_{\rm drive}(g_1\cdot x_{\rm ego})),$$

$$h_t=\phi(W_{\rm mix}h_{t-1}+W_{\rm drive}(g_t\cdot x_{\rm ego})),\qquad y_t=W_{\rm out}h_t.$$

The trace-feature construction contributes $12d_\rho^3$ hidden units per irrep. We apply

$$W_{\rm mix}h=W_{\rm in}(W_{\rm out}h)$$

without materializing a hidden-by-hidden matrix. When every irrep is included and every $\widehat{x}_{\rm ego}(\rho)$ is invertible, $y_t=(g_t\cdots g_1)\cdot x_{\rm allo}$ exactly, up to floating-point error.

In [ ]:
rng = np.random.default_rng(7)
x_allo_exact = rng.normal(size=G_exact.order)
x_ego_exact = random_invertible_encoding(G_exact, irreps_exact, seed=11)

params_exact = build_finite_group_rnn(
    G_exact,
    x_ego_exact,
    irrep_selection="all",
    materialize_mix=False,
)

print("hidden width:", params_exact.hidden_dim)
print("W_in shape:", params_exact.W_in.shape)
print("W_out shape:", params_exact.W_out.shape)
print("W_mix materialized:", params_exact.W_mix is not None)
assert params_exact.hidden_dim == exact_width

## 3. Exactness checks

The complete $n=2$ model is tested on translations, rotations about different axes, and mixed noncommuting sequences. All tokens are relative drives; the initial signal is supplied separately. This checks both the representation formula and the multiplication order $g_T\cdots g_1$.

In [ ]:
def rotation_index(group, matrix):
    matrix = np.asarray(matrix)
    return next(
        rotation
        for rotation in range(group.num_rotations)
        if np.array_equal(group.rotation_matrix(rotation), matrix)
    )


rotation_x_90 = rotation_index(
    G_exact,
    [[1, 0, 0], [0, 0, -1], [0, 1, 0]],
)
rotation_z_90 = rotation_index(
    G_exact,
    [[0, -1, 0], [1, 0, 0], [0, 0, 1]],
)

exact_sequences = {
    "translation": [G_exact.encode(1, 0, 0, 0)],
    "rotation": [G_exact.encode(0, 0, 0, rotation_x_90)],
    "mixed": [
        G_exact.encode(1, 0, 0, 0),
        G_exact.encode(0, 0, 0, rotation_z_90),
        G_exact.encode(0, 1, 0, 0),
        G_exact.encode(0, 0, 0, rotation_x_90),
    ],
}

for name, sequence in exact_sequences.items():
    result = rollout(params_exact, x_allo_exact, sequence)
    error = np.max(np.abs(result["predicted_outputs"] - result["true_outputs"]))
    print(f"{name:12s} max absolute error = {error:.3e}")
    assert error < 1e-10

## 4. Truncated spatial and orientation experiment

At $n=3$, the full group has 648 elements and the all-irrep construction would require 71,040 hidden units. We instead select at most six irreps under a 10,000-unit budget, ranked by Fourier power per hidden unit.

The allocentric signal is an off-center anisotropic Gaussian with a profile peaked at the identity orientation. Both choices matter: an isotropic signal copied uniformly over rotations cannot diagnose orientation errors.

In [ ]:
G = DiscreteSE3Group(n=3)
initial_pose = (1, 0, 2, 0)
x_allo = gaussian_landmark(
    G,
    center=initial_pose[:3],
    sigma=(0.4, 0.7, 1.0),
    orientation_weights=peaked_orientation_weights(G, rotation=initial_pose[3], floor=0.05),
)

all_irreps = G.irreps()
selected_irreps, selected_indices = select_irreps_by_power(
    all_irreps,
    x_allo,
    num_irreps=6,
    max_hidden_width=10_000,
    ranking="power_per_hidden",
)
x_ego = random_invertible_encoding(G, selected_irreps, seed=19)
params = build_finite_group_rnn(
    G,
    x_ego,
    irreps=all_irreps,
    x_allo=x_allo,
    irrep_selection="power",
    num_irreps=6,
    max_hidden_width=10_000,
    power_ranking="power_per_hidden",
    materialize_mix=False,
)

print(f"|G|={G.order}; all irreps={len(all_irreps)}")
print("selected global indices:", params.selected_irrep_indices)
print("selected dimensions:", [irrep.dim for irrep in params.irreps])
print("hidden width:", params.hidden_dim)

In [ ]:
spatial = spatial_marginal(G, x_allo)
plot_orthogonal_slices(spatial, center=initial_pose[:3], title="Allocentric spatial encoding")
plt.show()

plot_orientation_marginal(G, x_allo, title="Allocentric orientation encoding")
plt.show()

## 5. Translation, rotation, and mixed rollouts

We report three distinct errors:

- relative $\ell_2$ error of the complete group signal;
- periodic Euclidean error of the decoded spatial center;
- geodesic angle between decoded and target cubic rotations.

Separating them reveals whether truncation preserves pose even when signal shape or amplitude is imperfect.

In [ ]:
def summarize_rollout(name, sequence):
    result = rollout(params, x_allo, sequence)
    signal_errors = np.linalg.norm(
        result["predicted_outputs"] - result["true_outputs"], axis=1
    ) / np.linalg.norm(result["true_outputs"], axis=1)

    predicted_poses = []
    target_poses = []
    position_errors = []
    orientation_errors = []
    for predicted, state in zip(
        result["predicted_outputs"], result["cumulative_states"]
    ):
        predicted_pose = decode_pose(G, predicted)
        target_pose = transformed_pose(G, int(state), initial_pose)
        predicted_poses.append(predicted_pose)
        target_poses.append(target_pose)
        position_errors.append(
            np.sqrt(periodic_distance_squared(G.n, predicted_pose[:3], target_pose[:3]))
        )
        orientation_errors.append(
            rotation_error(G, predicted_pose[3], target_pose[3])
        )

    print(
        f"{name:16s} final signal={signal_errors[-1]:.3e}; "
        f"max position={max(position_errors):.3f}; "
        f"max rotation={np.degrees(max(orientation_errors)):.1f}°"
    )
    return {
        "rollout": result,
        "signal_errors": np.asarray(signal_errors),
        "position_errors": np.asarray(position_errors),
        "orientation_errors": np.asarray(orientation_errors),
        "predicted_poses": np.asarray(predicted_poses),
        "target_poses": np.asarray(target_poses),
    }


rotation_x_90 = rotation_index(
    G,
    [[1, 0, 0], [0, 0, -1], [0, 1, 0]],
)
rotation_z_90 = rotation_index(
    G,
    [[0, -1, 0], [1, 0, 0], [0, 0, 1]],
)

sequences = {
    "translation-only": [G.encode(1, 0, 0, 0)] * 4,
    "rotation-only": [G.encode(0, 0, 0, rotation_z_90)] * 4,
    "mixed": [
        G.encode(1, 0, 0, 0),
        G.encode(0, 0, 0, rotation_z_90),
        G.encode(0, 1, 0, 0),
        G.encode(0, 0, 0, rotation_x_90),
        G.encode(0, 0, 1, 0),
    ],
}
results = {name: summarize_rollout(name, sequence) for name, sequence in sequences.items()}

In [ ]:
mixed = results["mixed"]
steps = np.arange(1, len(mixed["signal_errors"]) + 1)

figure, axes = plt.subplots(1, 3, figsize=(12, 3.2), constrained_layout=True)
axes[0].plot(steps, mixed["signal_errors"], marker="o")
axes[0].set(xlabel="Step", ylabel="Relative signal error", title="Signal reconstruction")
axes[1].plot(steps, mixed["position_errors"], marker="o")
axes[1].set(xlabel="Step", ylabel="Position error", title="Decoded position")
axes[2].plot(steps, np.degrees(mixed["orientation_errors"]), marker="o")
axes[2].set(xlabel="Step", ylabel="Rotation error (degrees)", title="Decoded orientation")
plt.show()

plot_trajectory(mixed["target_poses"][:, :3], title="Target mixed trajectory")
plot_trajectory(mixed["predicted_poses"][:, :3], title="Decoded mixed trajectory")
plt.show()

In [ ]:
power = G.power_spectrum(x_allo)
retained_fraction = power[params.selected_irrep_indices].sum() / power.sum()
all_width = sum(hidden_width(irrep) for irrep in all_irreps)

print(f"retained Fourier power: {retained_fraction:.2%}")
print(f"selected hidden width: {params.hidden_dim:,}")
print(f"all-irrep hidden width: {all_width:,}")
print(f"width reduction: {1 - params.hidden_dim / all_width:.2%}")

## 6. Conclusions and limitations

- The all-irrep $n=2$ construction verifies exact finite-group computation for translations, cubic rotations, and mixed noncommuting sequences.
- The $n=3$ experiment is deliberately approximate. Fourier retention, full signal error, position error, and orientation error quantify different consequences of truncation.
- Orientation must be observable in the encoding. A rotation-invariant signal cannot validate rotational state.
- Hidden width scales as $12\sum_\rho d_\rho^3$. Higher-dimensional induced irreps make cost-aware selection essential in three dimensions.
- Factored recurrent mixing removes the $O(H^2)$ storage cost of a dense `W_mix`, but `W_in` and `W_out` still scale as $O(H|G|)$.
- This notebook constructs weights analytically. A learned discrete-$SE(3)$ experiment additionally requires a composition dataset that applies permutation actions without materializing the dense regular representation.